# 00 — Export locked Python model results for R
Run once with **Python (DSML CW2)** after both model notebooks finish successfully. This exports the already fitted benchmark's held-out residuals and locked classifier scores; it never fits a model or selects a threshold. Local bridge files contain sensitive row-level derived data: do not commit them or upload them. Run the R notebooks against the same export. Only load joblib files created by your own trusted project.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import average_precision_score

root = next((q for q in [Path.cwd(), *Path.cwd().parents] if (q/'data/processed/feature_contract.json').is_file()), None)
if root is None:
    raise FileNotFoundError('Run cleaning and validation first, then open this notebook inside the project.')
DATA = root/'data/processed'
OUT = root/'models/outputs'
BRIDGE = root/'statistical_analysis/private_exports'
BRIDGE.mkdir(parents=True, exist_ok=True)
def sha(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()
required = ['discordance_bundle.joblib','locked_classifiers.joblib','evaluation_lock.json','test_metrics_all.csv']
missing = [x for x in required if not (OUT/x).is_file()]
if missing:
    raise FileNotFoundError(f'Finish both model notebooks first. Missing: {missing}')
contract = json.loads((DATA/'feature_contract.json').read_text())
assert json.loads((DATA/'validation_report.json').read_text())['passed']
for name, expected in contract['output_sha256'].items():
    assert sha(DATA/name) == expected, f'Changed prepared input: {name}'
assert sha(root/'data/raw/diabetic_data.csv') == contract['source_sha256']
lock = json.loads((OUT/'evaluation_lock.json').read_text())
assert lock['contract_hash'] == sha(DATA/'feature_contract.json')
assert lock['model_hash'] == sha(OUT/'locked_classifiers.joblib')
assert lock['discordance_hash'] == sha(OUT/'discordance_bundle.joblib')
bundle = joblib.load(OUT/'discordance_bundle.joblib')
locked = joblib.load(OUT/'locked_classifiers.joblib')
assert bundle['contract_hash'] == locked['contract_hash'] == lock['contract_hash']
assert locked['primary_family'] == lock['primary_family']
T, G, ID = contract['target'], contract['group'], contract['id']
CLINICAL, CARE, DISC = [contract[k] for k in ['clinical_features','observed_care_features','planned_discordance_features']]
SETS = {'A':contract['feature_set_A'],'B':contract['feature_set_B'],'C':contract['feature_set_C_planned']}
print('Locked primary classifier:', locked['primary_family'])

Locked primary classifier: XGBoost


## Reconstruct the exact held-out feature inputs
The loading rules match the two Python modelling notebooks, including the categorical admission codes. Training discordance is already out-of-fold; validation and test discordance use the training-only benchmark.

In [2]:
def load_test():
    d = pd.read_csv(DATA/'test.csv', keep_default_na=False, na_values=['?', ''], dtype={G:str, ID:str})
    for col in contract['clinical_categorical']:
        if col in ['admission_type_id','admission_source_id']:
            d[col] = pd.to_numeric(d[col], errors='raise').map(lambda x: str(int(x)) if pd.notna(x) else np.nan).astype(object)
        else:
            d[col] = d[col].map(lambda x: str(x) if pd.notna(x) else np.nan).astype(object)
    for col in contract['clinical_numeric'] + CARE:
        d[col] = pd.to_numeric(d[col], errors='raise').astype(float)
    assert set(d[T]) == {0,1} and d[G].notna().all()
    return d
train, val = bundle['train'].copy(), bundle['validation'].copy()
test = load_test()
assert set(test[G]).isdisjoint(set(train[G]) | set(val[G]))
assert set(train[G]).isdisjoint(val[G])
test[DISC] = (test[CARE].to_numpy() - bundle['benchmark'].predict(test[CLINICAL])) / bundle['scale']
assert np.isfinite(test[DISC].to_numpy()).all()
assert len(locked['models']) == 12
# Verify every score against the metrics already saved by Notebook 02.
original = pd.read_csv(OUT/'test_metrics_all.csv')
score_cols = []
for i, ((family, label), entry) in enumerate(sorted(locked['models'].items())):
    model = entry['pipeline']
    X = test[SETS[label]]
    s = model.predict_proba(X)[:, list(model.classes_).index(1)] if hasattr(model,'predict_proba') else model.decision_function(X)
    assert np.isfinite(s).all()
    row = original[(original.model==family)&(original.feature_set==label)]
    assert len(row)==1
    assert np.isclose(average_precision_score(test[T],s), row.AP.iloc[0], atol=1e-9)
    assert np.isclose(entry['threshold'], row.threshold.iloc[0])
    key = f'model_{i+1:02d}'
    test[key+'_score'] = s
    test[key+'_pred'] = (s >= entry['threshold']).astype(int)
    score_cols.append(dict(model=family, feature_set=label, score=key+'_score', prediction=key+'_pred', threshold=float(entry['threshold'])))
print('All 12 AP values agree with the locked Python evaluation.')

All 12 AP values agree with the locked Python evaluation.


In [3]:
# Export only the columns needed; use split-local surrogate patient IDs.
# The export is pseudonymised/minimised, NOT guaranteed anonymous.
base = list(dict.fromkeys(CLINICAL + CARE + DISC + [T]))
for name, d in [('train',train),('validation',val),('test',test)]:
    extra = [x[k] for x in score_cols for k in ['score','prediction']] if name=='test' else []
    out = d[base+extra].copy()
    out.insert(0,'patient_cluster',pd.factorize(d[G], sort=False)[0]+1)
    assert d[G].notna().all()
    out.to_csv(BRIDGE/f'{name}.csv', index=False, na_rep='')
meta = dict(target=T, clinical_numeric=contract['clinical_numeric'], clinical_categorical=contract['clinical_categorical'],
            care=CARE, discordance=DISC, models=score_cols, primary_family=locked['primary_family'],
            contract_hash=lock['contract_hash'], model_hash=lock['model_hash'], discordance_hash=lock['discordance_hash'],
            rows={n:len(d) for n,d in [('train',train),('validation',val),('test',test)]})
(BRIDGE/'manifest.json').write_text(json.dumps(meta,indent=2))
print('Private R inputs exported:', BRIDGE)
print('Next: select R (DSML CW2) in notebooks 01–03.')

Private R inputs exported: /Users/monishachandrashekar/Desktop/DS&ML-CW2 /Data-science-and-Machine-Learning-Course-work/statistical_analysis/private_exports
Next: select R (DSML CW2) in notebooks 01–03.
